In [1]:
from camel.agents import ChatAgent
from camel.models import BaseModelBackend
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from datasets import load_dataset
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from librarian.llm_classifier import LLMClassifier
load_dotenv()

/home/yuan/.cache/pypoetry/virtualenvs/librarian-yhAyT7eh-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
dataset = load_dataset("basicv8vc/SimpleQA")

In [4]:
def create_openai_model(
    model_type: ModelType = ModelType.GPT_4O_MINI,
    model_config_dict: dict = {"temperature": 0.5},
) -> BaseModelBackend:
    """Create an OpenAI model."""
    load_dotenv()  # load the openai key from .env
    return ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=model_type,
        model_config_dict=model_config_dict,
    )

In [5]:
class LibrarianResponse(BaseModel):
    knowledge: str = Field(..., description="The retrieved knowledge.")
    reasoning: str = Field(..., description="The step-by-step reasoning process.")
    answer: str = Field(..., description="The final answer.")

system_message = \
"""You are a knowledge-responsible assistant who separates facts from reasoning.

**Step 1: Retrieve Knowledge**
First, find relevant, precise, and time-stamped knowledge related to the question. If you cannot find reliable knowledge, say so.

Only use well-known, verifiable sources (e.g., WHO, NASA, scientific papers). Do not answer yet.

Format your retrieved knowledge like this:
[
  {"source": "SourceName", "timestamp": "YYYY-MM-DD", "fact": "..." }
]

**Step 2: Reason Based on Retrieved Knowledge**
Use only the above facts to reason step by step. Do not use outside knowledge or assumptions. If knowledge is insufficient, say so.

Final Format:

Retrieved knowledge: ...

Step-by-step reasoning: ...

Answer: ... 
"""

librarian_agent = ChatAgent(model=create_openai_model(), system_message=system_message)
normal_agent = ChatAgent(model=create_openai_model())
classifer_agent = ChatAgent(model=create_openai_model())
llm_classifier = LLMClassifier(classifer_agent)

In [ ]:
librarian_correct = 0
plain_correct = 0

for dp in list(dataset["test"])[:50]:
    
    print("#### Question ####\n")
    print(dp["problem"])
    
    print("\n#### Librarian Agent Answer ####\n")
    librarian_agent.reset()
    librarian_response = librarian_agent.step(f"Question: {dp['problem']}", response_format=LibrarianResponse)
    print(eval(librarian_response.msgs[0].content)["answer"])
    
    print("\n#### Plain Agent Answer ####\n")
    normal_agent.reset()
    normal_response = normal_agent.step(dp["problem"])
    print(normal_response.msgs[0].content)
    
    print("\n#### Gold Answer ####\n")
    print(dp["answer"])
    
    print("\n#### In Comparison ####\n")
    
    librarian_grade = llm_classifier.grade(dp['problem'], dp['answer'], eval(librarian_response.msgs[0].content)["answer"])
    
    print("Librarian Agent Grade:", librarian_grade)
    
    plain_grade = llm_classifier.grade(dp['problem'], dp['answer'], normal_response.msgs[0].content)
    
    print("Plain Agent Grade:", plain_grade)

    
    if librarian_grade == "A":
        librarian_correct += 1
    if plain_grade == "A":
        plain_correct += 1
        
    
    print("\n")
    print("---------")
    print("\n")

#### Question ####

Who received the IEEE Frank Rosenblatt Award in 2010?

#### Librarian Agent Answer ####

Yoshua Bengio, Geoffrey Hinton, and Yann LeCun received the IEEE Frank Rosenblatt Award in 2010.

#### Plain Agent Answer ####

The IEEE Frank Rosenblatt Award in 2010 was awarded to Dr. Yoshua Bengio, Dr. Geoffrey Hinton, and Dr. Yann LeCun for their contributions to the field of neural networks and deep learning.

#### Gold Answer ####

Michio Sugeno

#### In Comparison ####

Librarian Agent Grade: {"grade":"B"}
Plain Agent Grade: {"grade":"B"}


---------


#### Question ####

Who was awarded the Oceanography Society's Jerlov Award in 2018?

#### Librarian Agent Answer ####

Dr. David R. C. H. Houghton was awarded the Oceanography Society's Jerlov Award in 2018.

#### Plain Agent Answer ####

The Oceanography Society's Jerlov Award in 2018 was awarded to Dr. David A. Siegel.

#### Gold Answer ####

Annick Bricaud

#### In Comparison ####

Librarian Agent Grade: {"grade":"B"}


In [ ]:
print(librarian_correct, plain_correct)

3 4
